<a href="https://colab.research.google.com/github/denisejroth/pixels-frames-transformers/blob/main/part2_vectors/05_cnns_for_images/notebooks/05_cnns_for_images_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pixels, Frames & Transformers
## Module 05: Convolutional Neural Networks for Images

**A Self-Paced Course in Computational Image and Video Analysis**  
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

In Part 1 you decided in advance which features matter: brightness, colorfulness, faces. A
convolutional neural network (CNN) learns its own features from labeled examples. This notebook
opens up the architecture behind a decade of computer vision: small filters that slide across an
image, layers that build edges into shapes and shapes into objects, and the training tricks that
stop a network from memorizing its data.

In this notebook you will:

- apply convolution filters and pooling by hand, to see what a CNN's first layers do
- build and train a small CNN on CIFAR-10, a benchmark with 50,000 labeled images
- watch a network overfit on too few images, and fight back with augmentation, dropout and early
  stopping
- look inside a trained network: its filters, its activation maps, and saliency maps
- train the same network on the protest photos and score it on your 40 coded images, where a few
  hundred labels turn out not to be enough

**Data.** [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) (Krizhevsky, 2009): 60,000 tiny
photos (32 × 32 pixels) in ten classes such as airplane, cat and truck, a standard benchmark for
teaching and research. Then the protest and King's Day photos from module 02, with your codes
from module 04 as the test set.

**Runtime.** A GPU makes this notebook much faster: *Runtime → Change runtime type → T4 GPU*. On a
CPU it still runs, but it automatically trains on a smaller part of CIFAR-10 for fewer epochs, so
your numbers will be lower.

## 0. Setup

We use **PyTorch** (`torch`) for the networks and **torchvision** for image datasets and
transformations; both are preinstalled in Colab. The course helper `course_data.py` loads the
protest photos and your codes, as in module 04.

If you saved your codes to Google Drive in module 04, Colab will ask for permission to read them
in section 6. If you downloaded them instead, upload `my_codes.csv` and `my_results.csv` with the
file browser on the left before you get there.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ON_GPU = DEVICE == "cuda"
print("Running on:", torch.cuda.get_device_name(0) if ON_GPU else "CPU (expect slower training)")

HEADERS = {"User-Agent": "PixelsFramesTransformers/1.0 (https://github.com/denisejroth/pixels-frames-transformers) course-notebook"}
COURSE_DATA_URL = "https://raw.githubusercontent.com/denisejroth/pixels-frames-transformers/main/shared/course_data.py"
if not Path("course_data.py").exists():
    response = requests.get(COURSE_DATA_URL, headers=HEADERS, timeout=30)
    response.raise_for_status()  # stop if the download failed, rather than saving an error page
    Path("course_data.py").write_text(response.text)
import course_data
from course_data import load_rgb

## 1. A one-minute refresher on neural networks

A neural network is a stack of simple steps, each with weights it can adjust:

- A **neuron** takes a weighted sum of its inputs and passes it through a simple non-linear
  function. The most common one, **ReLU**, keeps positive values and sets negative ones to 0.
- A **layer** is many neurons side by side; a **network** is layers stacked on top of each other.
- For classification, the last layer gives one score per class. The **loss** (here cross-entropy)
  measures how far those scores are from the right answer.
- **Training** repeats one move: show the network a small **batch** of examples, compute the
  loss, and nudge every weight in the direction that lowers it (**gradient descent**, with the
  gradients computed by **backpropagation**). One pass over all training data is an **epoch**.

If this is new to you, bonus module 6 of
[Bags, Vectors & Transformers](https://github.com/denisejroth/bags-vectors-transformers/tree/main/bonus)
builds a network step by step. What is new here is the kind of layer.

## 2. Convolutions by hand

Logistic regression on raw pixels treats an image like a bag of words: it has no idea which
pixels are neighbors. A **convolution** fixes that. A small grid of weights, the **filter**
(or kernel), slides across the image; at every position it multiplies the pixels under it by its
weights and adds them up. The result is a new image, a **feature map**, that is bright wherever
the image matches the filter's pattern.

Before a network learns its own filters, let's apply some classic hand-made ones to a portrait
from module 01. PyTorch expects images as tensors of shape (batch, channels, height, width).

In [ ]:
url = "https://raw.githubusercontent.com/unitedstates/images/gh-pages/congress/450x550/C000127.jpg"
Path("portrait.jpg").write_bytes(course_data.download(url).content)
portrait = load_rgb("portrait.jpg", 300)

gray = torch.tensor(np.asarray(portrait.convert("L")), dtype=torch.float32) / 255
image = gray[None, None]          # shape (1, 1, height, width): one image, one channel
print("Image tensor:", tuple(image.shape))

FILTERS = {
    "vertical edges": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],      # dark-to-bright from left to right
    "horizontal edges": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],    # dark-to-bright from top to bottom
    "blur": [[1 / 9] * 3] * 3,                                     # the average of a 3 x 3 patch
    "sharpen": [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}

def apply_filter(image, kernel):
    """Convolve a (1, 1, H, W) image with a 2-D kernel (a list of lists or a tensor)."""
    kernel = torch.as_tensor(kernel, dtype=torch.float32)[None, None]
    return F.conv2d(image, kernel, padding=kernel.shape[-1] // 2)

fig, axes = plt.subplots(1, len(FILTERS) + 1, figsize=(13, 3.5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("original")
for ax, (name, kernel) in zip(axes[1:], FILTERS.items()):
    out = apply_filter(image, kernel)[0, 0]
    limit = out.abs().max() if "edges" in name else None
    ax.imshow(out, cmap="gray", vmin=-limit if limit else None, vmax=limit)
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

The edge filters respond where brightness changes: the outline of the hair, the collar, the eyes.
A CNN's first layer learns dozens of filters like these by itself, in color, and deeper layers
combine their outputs into corners and textures, then into parts such as eyes or wheels, and
finally into whole objects.

> **✏️ Exercise 1**
>
> 1. Write a 3 × 3 filter that detects **diagonal** edges (running from the top left to the bottom
>    right) and apply it with `apply_filter`.
> 2. Write a **5 × 5 blur** (every weight 1/25) and compare it with the 3 × 3 blur.
> 3. What happens to the sum of an image's values when you apply a filter whose weights add up to 0,
>    such as the edge filters, in a region of even color?

In [ ]:
# Your code here

CNNs also **pool**: they summarize small blocks of a feature map, usually by keeping the maximum
of every 2 × 2 block. The map gets four times smaller, which saves computation and makes the
network a little less sensitive to exactly where in the image something appears.

In [ ]:
edges = apply_filter(image, FILTERS["vertical edges"]).abs()
pooled = F.max_pool2d(edges, kernel_size=2)
pooled_twice = F.max_pool2d(pooled, kernel_size=2)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, m, title in zip(axes, [edges, pooled, pooled_twice], ["edge map", "max-pooled once", "max-pooled twice"]):
    ax.imshow(m[0, 0], cmap="gray")
    ax.set_title(f"{title}\n{tuple(m.shape[-2:])}")
    ax.axis("off")
plt.show()

## 3. A small CNN for CIFAR-10

CIFAR-10 is small enough to train on in minutes and large enough for a CNN to shine. We load the
50,000 training images and the 10,000 test images (about 170 MB), turn them into tensors, and
**normalize** each color channel (subtract its average and divide by its spread), which helps
training.

We set aside part of the training images as a **validation set** to watch during training. On a
CPU, the notebook trains on a smaller sample of 10,000 images.

In [ ]:
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
to_tensor = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

cifar_train = datasets.CIFAR10(root="data", train=True, download=True, transform=to_tensor)
cifar_test = datasets.CIFAR10(root="data", train=False, download=True, transform=to_tensor)
CLASSES = cifar_train.classes

order = torch.randperm(len(cifar_train), generator=torch.Generator().manual_seed(SEED)).tolist()
n_val = min(5000, len(cifar_train) // 10)
N_TRAIN = 45_000 if ON_GPU else 10_000
val_idx, train_idx = order[:n_val], order[n_val:n_val + N_TRAIN]

def loader(dataset, shuffle=False, batch_size=128):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=2, pin_memory=ON_GPU)

train_loader = loader(Subset(cifar_train, train_idx), shuffle=True)
val_loader = loader(Subset(cifar_train, val_idx))
test_loader = loader(cifar_test)
print(f"training: {len(train_idx)}  validation: {len(val_idx)}  test: {len(cifar_test)}")

def unnormalize(x, mean=CIFAR_MEAN, std=CIFAR_STD):
    """Turn a normalized (3, H, W) tensor back into a displayable (H, W, 3) array."""
    x = x.cpu() * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axes.flat, train_idx[:16]):
    x, y = cifar_train[i]
    ax.imshow(unnormalize(x))
    ax.set_title(CLASSES[y], fontsize=9)
    ax.axis("off")
plt.show()

Now the network. Three blocks of convolution, ReLU and pooling, each with more filters than the
last, followed by **global average pooling** (the average of each feature map, whatever its size)
and one linear layer that turns those averages into ten class scores. Because of the global
pooling, the same network accepts images of any size, which we will need in section 6.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10, dropout=0.0):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                      # global average pooling
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

model = SmallCNN()
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\n{n_params:,} weights to learn")

> **✏️ Exercise 2**
>
> Trace an image through the network. Pass a dummy batch `torch.zeros(1, 3, 32, 32)` through
> `model.features` **one layer at a time** and print the shape after each layer. Where does the
> image get smaller, and where does it get "deeper" (more channels)? Then do the same with a
> 96 × 96 image.

In [ ]:
# Your code here

Next, a training loop. `train_model` runs a number of epochs; after each one it measures loss and
accuracy on the training and validation data. It also keeps a copy of the weights from the epoch
with the lowest validation loss and restores them at the end: **early stopping** in its simplest
form.

In [ ]:
def run_epoch(model, data_loader, loss_fn, optimizer=None):
    """One pass over the data. Trains if an optimizer is given; returns (loss, accuracy)."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in data_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            loss = loss_fn(logits, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            n += len(y)
    return total_loss / n, correct / n

def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3, class_weights=None):
    """Train with Adam; keep the weights from the epoch with the lowest validation loss."""
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss(weight=None if class_weights is None else class_weights.to(DEVICE))
    history, best_loss, best_state = [], float("inf"), None
    for epoch in range(1, epochs + 1):
        start = time.time()
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                        "train_acc": train_acc, "val_acc": val_acc})
        if val_loss < best_loss:
            best_loss, best_state = val_loss, {k: v.detach().clone() for k, v in model.state_dict().items()}
        print(f"epoch {epoch:2d}  train loss {train_loss:.3f}, acc {train_acc:.2f}  |  "
              f"validation loss {val_loss:.3f}, acc {val_acc:.2f}  ({time.time() - start:.0f}s)")
    model.load_state_dict(best_state)
    return pd.DataFrame(history)

def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for ax, metric in zip(axes, ["loss", "acc"]):
        ax.plot(history["epoch"], history[f"train_{metric}"], marker="o", label="training")
        ax.plot(history["epoch"], history[f"val_{metric}"], marker="o", label="validation")
        ax.set_xlabel("epoch")
        ax.set_ylabel("accuracy" if metric == "acc" else "loss")
        ax.legend()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

@torch.no_grad()
def predict(model, data_loader):
    """Return (predicted classes, class probabilities) for every item in a loader."""
    model.eval()
    probs = torch.cat([model(x.to(DEVICE)).softmax(dim=1).cpu() for x, _ in data_loader])
    return probs.argmax(dim=1).numpy(), probs.numpy()

In [ ]:
torch.manual_seed(SEED)
EPOCHS = 8 if ON_GPU else 4
cnn = SmallCNN()
history = train_model(cnn, train_loader, val_loader, epochs=EPOCHS)
plot_history(history, "Small CNN on CIFAR-10")

The test set has never been used for training or for choosing anything, so it gives an honest
estimate. A **confusion matrix** shows which classes the network mixes up.

In [ ]:
test_pred, test_prob = predict(cnn, test_loader)
test_true = np.array(cifar_test.targets)
print(f"Test accuracy: {(test_pred == test_true).mean():.2f}  (guessing would give 0.10)")

fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay(confusion_matrix(test_true, test_pred), display_labels=CLASSES).plot(
    ax=ax, colorbar=False, xticks_rotation=45)
plt.tight_layout()
plt.show()

A few minutes of training with about 95,000 weights gets well beyond chance, and the confusions
make sense: cats and dogs, cars and trucks, deer and horses. Larger networks trained for longer
reach over 90% on CIFAR-10. But notice what made this possible: **tens of thousands of labeled
examples**.

## 4. Overfitting, and how to fight it

What if we only have 1,000 labeled images, a more realistic number for a social science project?
Let's train the same network on 1,000 CIFAR-10 images for 20 epochs and watch the two curves.

In [ ]:
small_idx = train_idx[:1000]
small_loader = loader(Subset(cifar_train, small_idx), shuffle=True, batch_size=64)

torch.manual_seed(SEED)
cnn_small = SmallCNN()
history_small = train_model(cnn_small, small_loader, val_loader, epochs=20)
plot_history(history_small, "1,000 training images, no defenses")

The training loss keeps falling while the validation loss bottoms out and climbs again: the
network has started to memorize its 1,000 images instead of learning what makes a cat a cat.
That is **overfitting**. Three standard defenses:

- **Data augmentation**: show slightly different versions of each image in every epoch (shifted,
  cropped, mirrored), so memorizing exact pixels no longer pays. Only use changes that keep the
  label true: a mirrored cat is still a cat, but a mirrored traffic sign may not be the same sign.
- **Dropout**: during training, randomly switch off part of a layer's inputs, so the network
  can't depend on any single one.
- **Early stopping**: keep the weights from the best validation epoch (our `train_model` already
  does this).

> **✏️ Exercise 3**
>
> Train a fresh `SmallCNN(dropout=0.3)` on the **same 1,000 images**, this time with data
> augmentation: a random crop with 4 pixels of padding and a random horizontal flip.
>
> 1. Create a second CIFAR-10 training dataset with the transform below and a `Subset` with
>    `small_idx`.
> 2. Train for 20 epochs and plot the history.
> 3. Compare the best validation accuracy with and without the defenses. Do the curves still
>    diverge?
>
> ```python
> augment = transforms.Compose([
>     transforms.RandomCrop(32, padding=4),
>     transforms.RandomHorizontalFlip(),
>     transforms.ToTensor(),
>     transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
> ])
> ```

In [ ]:
# Your code here

## 5. Looking inside the network

A trained CNN is not a black box in the literal sense: every weight can be inspected. Let's look
at what the network trained on CIFAR-10 learned.

**First-layer filters.** Each of the 32 filters in the first layer is a 3 × 3 × 3 block of
weights, which we can show as a tiny color image.

In [ ]:
weights = cnn.features[0].weight.detach().cpu()          # (32, 3, 3, 3)
fig, axes = plt.subplots(4, 8, figsize=(8, 4))
for ax, w in zip(axes.flat, weights):
    w = (w - w.min()) / (w.max() - w.min() + 1e-8)
    ax.imshow(w.permute(1, 2, 0), interpolation="nearest")
    ax.axis("off")
fig.suptitle("The 32 learned first-layer filters")
plt.show()

With only 3 × 3 pixels they are hard to read, but you can see color blobs and light-dark
contrasts in different directions, much like the hand-made filters of section 2.

**Activation maps** show what each filter responds to in a particular image.

In [ ]:
x, y = cifar_test[3]
with torch.no_grad():
    maps = cnn.features[:2](x[None].to(DEVICE))[0].cpu()     # after the first convolution and ReLU

fig, axes = plt.subplots(2, 9, figsize=(13, 3.2))
axes[0, 0].imshow(unnormalize(x))
axes[0, 0].set_title(CLASSES[y], fontsize=9)
axes[1, 0].axis("off")
for ax, m in zip(list(axes[0, 1:]) + list(axes[1, 1:]), maps[:16]):
    ax.imshow(m, cmap="viridis")
for ax in axes.flat:
    ax.axis("off")
plt.show()

**Saliency maps** ask which pixels the prediction is most sensitive to: we compute the gradient
of the predicted class's score with respect to every input pixel. Bright areas are pixels where a
small change would move the score most.

A caution before you use these in a paper: saliency maps are easy to over-read. Some popular
methods produce similar-looking maps even for a network with random weights (Adebayo et al.,
2018). Treat them as a hint for where to look, not as an explanation of the model's reasoning.

In [ ]:
def saliency(model, x):
    """Gradient of the predicted class score with respect to the input pixels (max over channels)."""
    model.eval()
    x = x[None].to(DEVICE).requires_grad_(True)
    scores = model(x)
    predicted = scores.argmax(dim=1).item()
    scores[0, predicted].backward()
    return x.grad.abs().amax(dim=1)[0].cpu().numpy(), predicted

fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for col, i in enumerate(range(6)):
    x, y = cifar_test[i]
    sal, predicted = saliency(cnn, x)
    axes[0, col].imshow(unnormalize(x))
    axes[0, col].set_title(f"true: {CLASSES[y]}\npredicted: {CLASSES[predicted]}", fontsize=8)
    axes[1, col].imshow(sal, cmap="hot")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

> **✏️ Exercise 4**
>
> Find six test images the network got **wrong** (use `test_pred` and `test_true`) and show each
> image with its saliency map, as above. Does the network seem to look at the object, at the
> background, or at nothing in particular? What would you conclude, and what would you *not*
> conclude, from these maps?

In [ ]:
# Your code here

## 6. The same network on protest photos

Back to the research task from module 04: is an image a protest or not? We load the same split as
before: your 40 coded images are the test set, and every other protest and King's Day photo is a
training image, labeled by the Commons category it came from.

The images are much larger than CIFAR-10, so we scale them down to 96 × 96 pixels. Because there
are far more protest photos than King's Day photos, we weight the loss so mistakes on the smaller
class count more, as with `class_weight="balanced"` in module 04.

In [ ]:
train_table, test_table = course_data.protest_train_test()

IMG_MEAN, IMG_STD = (0.5, 0.5, 0.5), (0.25, 0.25, 0.25)
plain = transforms.Compose([transforms.Resize(110), transforms.CenterCrop(96),
                            transforms.ToTensor(), transforms.Normalize(IMG_MEAN, IMG_STD)])

class ImageTable(Dataset):
    """Images from a list of file paths, with integer labels."""
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = list(paths), list(labels), transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.transform(load_rgb(self.paths[i], 256)), int(self.labels[i])

fit_part, val_part = train_test_split(train_table, test_size=0.2, stratify=train_table["category_label"], random_state=SEED)
counts = np.bincount(fit_part["category_label"], minlength=2)
class_weights = torch.tensor(len(fit_part) / (2 * counts), dtype=torch.float32)
print("training images per class (King's Day, protest):", counts.tolist())

protest_fit = loader(ImageTable(fit_part["local_path"], fit_part["category_label"], plain), shuffle=True, batch_size=32)
protest_val = loader(ImageTable(val_part["local_path"], val_part["category_label"], plain), batch_size=32)
protest_test = loader(ImageTable(test_table["local_path"], test_table["protest"], plain), batch_size=32)

In [ ]:
torch.manual_seed(SEED)
protest_cnn = SmallCNN(n_classes=2, dropout=0.3)
history_protest = train_model(protest_cnn, protest_fit, protest_val, epochs=20, class_weights=class_weights)
plot_history(history_protest, "Small CNN from scratch on protest photos")

test_pred_protest, _ = predict(protest_cnn, protest_test)
score = course_data.log_score("small CNN from scratch", test_table["protest"], test_pred_protest, module="05")
print(score)
course_data.show_scores()

Compare the CNN with the random forest on handcrafted features from module 04 in your scoreboard.
With a few hundred training images, many of them filed under a category that doesn't describe
what they show, a network that has to learn everything from pixels rarely does better than
handcrafted features, and often does worse. Validation accuracy also jumps around from epoch to
epoch: with so few validation images, every single image moves it by several points.

> **✏️ Exercise 5**
>
> Give the protest CNN a better chance with **augmentation**: train a fresh `SmallCNN(n_classes=2,
> dropout=0.3)` with random crops and flips, plus `transforms.ColorJitter(brightness=0.3,
> contrast=0.3, saturation=0.3, hue=0.1)`. The color jitter makes it harder to rely on color alone,
> such as the orange of King's Day.
>
> Log the result on your codes as `"small CNN from scratch, augmented"` and compare it with the
> other scores. Did augmentation help, and by how much would you trust the difference with 40 test
> images?

In [ ]:
# Your code here

> **✏️ Exercise 6**
>
> Why does a network that does so well on CIFAR-10 struggle with the protest photos? List at least
> three differences between the two tasks, and say for each what would help.

*Your answer here*

## Wrap-up

You have seen that:

- a convolution slides a small filter over an image, and pooling summarizes its output, so a CNN
  can find a pattern wherever it appears;
- a CNN learns its own filters, from simple edges and colors in the first layer to whole objects
  in later ones;
- with tens of thousands of labeled images, a small CNN learns a lot in minutes;
- with a thousand or fewer, it overfits, and augmentation, dropout and early stopping help but
  don't replace data;
- filters, activation maps and saliency maps let you look inside, but saliency maps are hints,
  not explanations;
- on the protest task, a CNN trained from scratch has too few, too noisy labels to beat
  handcrafted features.

**Next:** module 06 borrows a network that someone else trained on millions of images, and turns
every image into a vector that makes a few hundred labels go a long way.